In [3]:
import pandas as pd
import numpy as np
import os

from tqdm import tqdm
import glob
import ast
import re

import faiss
from uuid import uuid4

from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.docstore.document import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

import strip_markdown
import itertools
import json

# ======= config path =======
BASE_DIR = '/project/lt200304-dipmt/paweekorn'

RETRIEVER = "bge-m3"
MODEL_PATH = f"{BASE_DIR}/superai-6/RAG/models/{RETRIEVER}"
RESULT_DIR = f"./vectorstore/{RETRIEVER}"

## Overview

In [4]:
knowledge_files = glob.glob("./data/knowledge_base/*/*")

markdown_text = []
for knowledge in tqdm(knowledge_files):
    with open(knowledge, "r") as f:
        markdown_text.append(f.read())
        f.close()

markdown_text[0]

100%|██████████| 118/118 [00:00<00:00, 2503.62it/s]


'# พัลส์เกียร์ แบตสำรอง 30,000 mAh (PulseGear Power Bank 30000mAh)\n\nรหัสสินค้า: PG-PB-002\nแบรนด์: พัลส์เกียร์ (PulseGear) — แบรนด์พันธมิตร\nหมวดหมู่: แบตสำรอง\nราคา: ฿2,290\nสถานะ: มีสินค้า\nวันที่อัปเดต: 1 มีนาคม 2569\n\n## รายละเอียดสินค้า\n\nเมื่อการเดินทางยาวนานกว่าหนึ่งวัน PulseGear แบตสำรอง 30,000 mAh คือเพื่อนร่วมทางที่ไว้วางใจได้ ความจุ 30,000 mAh รองรับการชาร์จสมาร์ทโฟนได้มากกว่า 6-7 ครั้ง หรือชาร์จแล็ปท็อปขนาดกลางได้ 2-3 ครั้ง ด้วย USB-C Power Delivery สูงสุด 100W พอร์ต USB-C หลักสามารถชาร์จแล็ปท็อปไฮเอนด์ที่ต้องการกำลังไฟสูงได้โดยตรง\n\nพอร์ต USB-C สอง ช่องและ USB-A หนึ่งช่องรองรับการชาร์จพร้อมกันสามอุปกรณ์ จอ LED แสดงเปอร์เซ็นต์แบตเตอรี่แบบตัวเลข ให้รู้สถานะได้ทันที น้ำหนักที่ 680 กรัมถือว่าเหมาะสมสำหรับความจุในระดับนี้\n\n**หมายเหตุสำคัญสำหรับการจัดส่ง:** เนื่องจากแบตสำรองนี้มีความจุมากกว่า 20,000 mAh ตามระเบียบการขนส่งทางอากาศสากล จึง**ไม่สามารถจัดส่งทางอากาศได้** พื้นที่ที่การจัดส่งต้องอาศัยขนส่งทางอากาศ (เช่น เกาะห่างไกล) อาจใช้เวลาเพิ่มเติม 3-5 วันทำการ โปรดตรวจสอบต

In [9]:
def refactored_chunking(markdown_text, chunk_size=800, chunk_overlap=150):
    """
    Refactored to preserve structural integrity for product RAG.
    Note: Removed 'strip_markdown' to keep table and header context.
    """
    
    # 1. Setup Splitter with smarter separators
    # We prioritize splitting by Headers (##) and Paragraphs (\n\n) 
    # to keep tables and lists together as much as possible.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n\n", "\n", " ", ""],
        strip_whitespace=True
    )
    
    # 2. Create Chunks
    # We pass the raw markdown. LLMs understand Markdown tables 
    # much better than stripped, messy text.
    chunks = text_splitter.split_text(markdown_text)
    
    # 3. Create Document objects with metadata
    # Adding 'source' or 'product_name' helps the RAG retrieve better.
    documents = [
        Document(
            page_content=t, 
            metadata={"char_count": len(t)}
        ) for t in chunks
    ]
    
    return documents

# Usage Improvement: Process multiple files
documents = []
# Assuming markdown_text is a list of strings
for content in markdown_text:
    doc_chunks = refactored_chunking(content, chunk_size=1200, chunk_overlap=200)
    documents.extend(doc_chunks)

print(f"Total Chunks Created: {len(all_documents)}")
if documents:
    print("-" * 30)
    print("Sample Chunk Content:")
    print(documents[1].page_content)

Total Chunks Created: 473
------------------------------
Sample Chunk Content:
## สเปคสินค้า

| รายการ | รายละเอียด |
|--------|-----------|
| ความจุ | 30,000 mAh |
| Output USB-C พอร์ต 1 | USB-C PD สูงสุด 100W |
| Output USB-C พอร์ต 2 | USB-C PD สูงสุด 18W |
| Output USB-A | USB-A 5V/2.4A (12W) |
| กำลังชาร์จรวมสูงสุด (Multi-port) | 100W |
| Input (ชาร์จตัว Power Bank) | USB-C 65W |
| เวลาชาร์จเต็ม (Input 65W) | ประมาณ 5.5 ชั่วโมง |
| จอแสดงผล | LED แสดงเปอร์เซ็นต์แบตเตอรี่ |
| น้ำหนัก | 680 กรัม |
| ขนาด | 165 × 75 × 27 มม. |
| ข้อจำกัดการขนส่ง | **ไม่สามารถจัดส่งทางอากาศได้** (ความจุ >20,000 mAh) |
| สี | สีดำ (Midnight Black) |

## สิ่งที่อยู่ในกล่อง

- PulseGear Power Bank 30,000mAh ×1
- สาย USB-C to USB-C 0.6m ×1
- คู่มือการใช้งาน (ภาษาไทย/English)

*หมายเหตุ: ไม่รวมหัวชาร์จ*


## Embedding

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name=MODEL_PATH)
d_model = len(embedding_model.embed_query('Hello World'))
d_model

1024

In [10]:
uuids = [str(uuid4()) for _ in range(len(documents))]

index_flat = faiss.IndexFlatL2(d_model)

# make it into a gpu index
vector_store = FAISS(
    embedding_function=embedding_model,
    index=index_flat,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
doc_ids = vector_store.add_documents(documents=documents, ids=uuids)
print(doc_ids[:5])

['645b490f-718e-4f0d-8bec-5073aafd1857', '929c124d-97dc-4571-87d4-fe89e6d84288', '668d1b82-54f4-4522-938e-93adbb35cd17', '41ed0437-b7fb-4078-84fe-bf3d6d37613d', 'd311cd38-e9b3-457e-a66f-b0988c3a5f2e']


In [11]:
# Move the index to CPU before saving
os.makedirs(f"{RESULT_DIR}", exist_ok=True)
cpu_index = faiss.index_gpu_to_cpu(vector_store.index)
faiss.write_index(cpu_index, f"{RESULT_DIR}/index.faiss")

# Re-create the FAISS vector store from the saved index, docstore, and index_to_docstore_id
vector_store_loaded = FAISS(
    embedding_function=embedding_model,
    index=cpu_index,
    docstore=vector_store.docstore,
    index_to_docstore_id=vector_store.index_to_docstore_id,
)

# Now you can save the other components using save_local
vector_store_loaded.save_local(RESULT_DIR)